# Anomaly Detection with Unsupervised Learning

In this asssignment, you will use `netml` to perform unsupervised learning on a dataset that contains attack traffic. Unsupervised learning refers to the process where models are trained without training or "supervision". They can be used to detect structure in data, including the process of detecting outliers.

In the below examples, an OCSVM model is trained by demo traffic included in the library, and tested by labels in a CSV file, (both provided by the University of New Brunswick's Intrusion Detection Systems dataset).

## Setup: Clone the Data Repository

Before starting, you need to clone the course data repository to access the datasets:

```bash
git clone https://github.com/noise-courses/data.git
```

This repository contains all the packet captures and labels needed for this assignment in the `data/anomaly-detection/` directory.

## Part 1: Warmup

In [1]:
from sklearn.model_selection import train_test_split

from netml.ndm.model import MODEL
from netml.ndm.ocsvm import OCSVM
from netml.pparser.parser import PCAP

RANDOM_STATE = 42

pcap = PCAP(
    'data/anomaly-detection/demo.pcap',
    flow_ptks_thres=2,
    random_state=42,
    verbose=0,
)

# extract flows from pcap
pcap.pcap2flows(q_interval=0.9)

# label each flow (optional)
pcap.label_flows(label_file='data/anomaly-detection/demo.csv')

# extract features from each flow via IAT
pcap.flow2features('IAT', fft=False, header=False)

# load data
(features, labels) = (pcap.features,
                      pcap.labels)

# split train and test sets
(
    features_train,
    features_test,
    labels_train,
    labels_test,
) = train_test_split(features, labels, test_size=0.33, random_state=RANDOM_STATE)

# create detection model
ocsvm = OCSVM(kernel='rbf', nu=0.5, random_state=RANDOM_STATE)
ocsvm.name = 'OCSVM'
ndm = MODEL(ocsvm, score_metric='auc', verbose=0, random_state=RANDOM_STATE)

# train the model from the train set
ndm.train(features_train)

# evaluate the trained model
ndm.test(features_test, labels_test)

# stats
print(ndm.train.tot_time, ndm.test.tot_time, ndm.score)

0.0 0.0 0.632


## Part 2: Explore Other Anomaly Detection Models

The `netml` library supports a number of anomaly detection models, including:

* Autoencoder
* Gaussian Mixture Model
* Independent Forest
* Kernel Density Estimation
* One-Class SVM (OCSVM) (as shown above)
* Principal Compoenent Analysis

Try the above example on two (or more) of the following models. Which tend to perform better?

In [2]:
from netml.ndm.model import MODEL
from netml.ndm.gmm import GMM
from netml.ndm.ae import AE   
from netml.ndm.iforest import IF
from netml.ndm.pca import PCA

models = [
    GMM(n_components=1, random_state=42), 
    AE(epochs=100, batch_size=32, random_state=42, verbose=0), 
    IF(n_estimators=100, random_state=42, verbose=0), 
    PCA(n_components=None, random_state=RANDOM_STATE)
]

for det in models:
    det.name = type(det).__name__
    ndm = MODEL(det, score_metric='auc', verbose=0, random_state=42)
    ndm.train(features_train)
    ndm.test(features_test, labels_test)
    print(det.name, ndm.score)

GMM 0.5680000000000001


AE 0.624
IF 0.40800000000000003
PCA 0.6160000000000001


Independent Forest was the worst as it performed literally worse than random (0.41), and GMM was slightly above random (0.57). PCA (0.62) and AE did the best. AE changes every time I run it (I got 0.55 and 0.62) because its training is random, so it's about tied with PCA. I suspect PCA performed well because the features from the packet capture are likely very correlated, PCA is able to find flows that deviate from standard heavily, allowing for greater accuracy.

## Part 3: Anomaly Detection for Activity Detection

Now that you have some basic experience with using the `netml` library for anomaly detection, your assignment is to apply anomaly detection on a different dataset and problem: activity detection.

When users interact with smart home devices in a connected home setting, these devices may generate traffic that differs from idle behavior (e.g., an increase in traffic volume).  Your task is to train anomaly detection models to distinguish *idle* traffic from *device interaction*.

We have provided several smart home device traces in `data/anomaly-detection/`:

* **google_home.pcap** (74 MB) - Google Home device with multiple activities including voice queries and music playing
* **google_home_labels.txt** - Labels showing timestamps for voice queries ("asked 'What is the time'"), music playing activity ("started playing music" / "stopped playing music"), with idle periods between activities
* **nestcam_live.pcap** (9.7 MB) - Nest camera with livestream and motion detection events
* **nestcam_live_labels.txt** - Labels showing "livestream" and "motion" events with timestamps
* **tplink_switch.pcap** (52 KB) - TP-Link smart switch traffic

Your task will be to:

1. Use netml to construct flows from these traces
2. Use netml to label the flows (you will probably have to work with the provided label files or create your own labeling scheme)
3. Evaluate one or more anomaly detection models to detect device activity (e.g., music playing, voice queries) vs. idle traffic

## Part 3.1: Construct the Flows

Construct flows from the provided pcaps in `data/anomaly-detection/`. You can choose to work with one or more of the device traces:
- **google_home.pcap**: Contains voice queries and music playing activities with idle periods in between
- **nestcam_live.pcap**: Contains livestream and motion detection events
- **tplink_switch.pcap**: Smart switch traffic

The label files (google_home_labels.txt, nestcam_live_labels.txt) provide timestamps showing when activities occurred. You will need to use these timestamps to identify which flows correspond to active behavior vs. idle behavior.

Hint: You may want to start with a single device trace (e.g., google_home.pcap) and focus on distinguishing between one type of activity (e.g., music playing) and idle periods.

In [3]:
google_home_pcap = PCAP(
    'data/anomaly-detection/google_home.pcap',
    flow_ptks_thres=2,
    random_state=42,
    verbose=0,
)

nestcam_live_pcap = PCAP(
    'data/anomaly-detection/nestcam_live.pcap',
    flow_ptks_thres=2,
    random_state=42,
    verbose=0,
)

google_home_pcap.pcap2flows(q_interval=0.9)
nestcam_live_pcap.pcap2flows(q_interval=0.9)



I'm using the nest cam trace for Part 3 because its labels line up cleanly with the traffic.

**How I aligned the labels:** the label file only has clock times (like 16:10:00) with no date or time zone, while the pcap timestamps are Unix time in UTC. The capture runs 20:10 to 20:30 UTC on 2016-07-29 and the labels run 16:10 to 16:28, so the labels are in EDT (UTC-4). I added the date 2016-07-29 and the -4 hour offset to every label time to turn it into a Unix timestamp, so it's on the same time base as the packets. You can see it lines up below: the trace starts at 16:10:03 EDT, right when the first livestream starts, and the traffic goes way up at each livestream time and drops at each motion time.

In [4]:
import datetime
import numpy as np

# pcap2flows only gives 2 flows for the nest cam (one long connection each way)
# so i split each flow into 10 second windows and treat each window as a flow
WINDOW = 10

nestcam_flows = []
for fid, pkts in nestcam_live_pcap.flows:
    chunks = {}
    for pkt in pkts:
        chunks.setdefault(int(float(pkt.time) // WINDOW), []).append(pkt)
    for k in sorted(chunks):
        if len(chunks[k]) >= 2:
            nestcam_flows.append((fid, chunks[k]))

nestcam_live_pcap.flows = nestcam_flows
print('flows before split: 2, after split:', len(nestcam_flows))


flows before split: 2, after split: 240


In [5]:
# the label file only has clock times, so i add the capture date and the time zone
# the pcap is in UTC and the labels line up with UTC-4 (EDT, it was recorded in july)
edt = datetime.timezone(datetime.timedelta(hours=-4))

events = []
with open('data/anomaly-detection/nestcam_live_labels.txt') as f:
    for line in f:
        t, name = line.strip().split(', ')
        ts = datetime.datetime.strptime('2016-07-29 ' + t, '%Y-%m-%d %H:%M:%S').replace(tzinfo=edt).timestamp()
        events.append((ts, name))

first = min(float(pkts[0].time) for fid, pkts in nestcam_flows)
last = max(float(pkts[-1].time) for fid, pkts in nestcam_flows)
print('trace:', datetime.datetime.fromtimestamp(first, edt), 'to', datetime.datetime.fromtimestamp(last, edt))
for ts, name in events:
    print(datetime.datetime.fromtimestamp(ts, edt).time(), name)


trace: 2016-07-29 16:10:03.220967-04:00 to 2016-07-29 16:30:03.398886-04:00
16:10:00 livestream
16:12:20 motion
16:14:00 livestream
16:16:20 motion
16:18:00 livestream
16:20:45 motion
16:22:00 livestream
16:24:15 motion
16:26:00 livestream
16:28:20 motion


## Part 3.2: Label the Flows

You will need to label your flows based on the provided label files. The label files show timestamps for when activities occurred. For example, google_home_labels.txt contains entries like:

```
1:51:00, started playing music
1:53:00, stopped playing music
```

You will need to create a labeling scheme that associates each flow with either "normal" (idle) or "anomaly" (active) behavior based on these timestamps. This may require:
- Parsing the timestamp format in the label files
- Mapping flow timestamps to activity windows
- Creating binary labels (0 = idle, 1 = active) or categorical labels based on activity type

You can either work with netml's label_flows function or create your own labeling function.

In [6]:
# active = from each livestream until the next motion event, everything else is idle
windows = [(events[i][0], events[i + 1][0]) for i in range(0, len(events), 2)]

def label_flow(pkts):
    start = float(pkts[0].time)
    for a, b in windows:
        if a <= start < b:
            return 1
    return 0

nestcam_labels = np.array([label_flow(pkts) for fid, pkts in nestcam_flows])
print('active:', nestcam_labels.sum(), 'idle:', (nestcam_labels == 0).sum())


active: 146 idle: 94


**Labeling scheme:** a flow is active (1) if it starts between a livestream event and the next motion event, otherwise it's idle (0). I picked those windows because that's when the camera is actually streaming video and the traffic jumps. Each flow is a 10 second window, so a few windows cross a boundary (like the one from 16:20:40 to 16:20:50, since motion happens at 16:20:45). For those I just go by the flow's start time, so a window that starts inside an active period counts as active even if it ends after it.

## Part 3.3: Train and Evaluate a Model

Select one of the unsupervised models in the `netml` library and train and evaluate your model to detect device interactions (e.g., music playing, voice queries) vs. idle traffic.

Consider:
- Which features are most useful for distinguishing active from idle behavior?
- How does your model's performance compare to the intrusion detection task in Part 1?
- What are the challenges in detecting different types of activities (e.g., voice queries vs. music playing)?

In [7]:
for feat in ['IAT', 'STATS']:
    nestcam_live_pcap.flow2features(feat, fft=False, header=False)
    X = np.array(nestcam_live_pcap.features)

    idle = X[nestcam_labels == 0]
    active = X[nestcam_labels == 1]

    # train only on idle flows, test on held out idle flows + all the active ones
    idle_train, idle_test = train_test_split(idle, test_size=0.3, random_state=RANDOM_STATE)
    X_test = np.vstack([idle_test, active])
    y_test = np.array([0] * len(idle_test) + [1] * len(active))

    pca = PCA(n_components=None, random_state=RANDOM_STATE)
    pca.name = 'PCA'
    ndm = MODEL(pca, score_metric='auc', verbose=0, random_state=RANDOM_STATE)
    ndm.train(idle_train)
    ndm.test(X_test, y_test)
    print(feat, 'PCA AUC:', ndm.score)


IAT PCA AUC: 0.9324515824279641
STATS PCA AUC: 1.0


**Results:** PCA trained only on idle flows gets an AUC of about 0.93 with IAT features and 1.0 with STATS features (byte rate, packet rate, packet size stats). That means it can basically perfectly tell when the camera is livestreaming vs idle, which is way better than the intrusion detection task in Part 1 (around 0.6). It makes sense because livestreaming sends a ton more data than idle, so the volume features (STATS) separate them really easily. IAT only looks at packet timing so it does a bit worse. This would probably be harder for something like a short voice query on the google home, since that's only a small burst of traffic.

**AI use:** I used Claude (AI) to help with Parts 2 and 3: how to call the netml models in Part 2, and in Part 3 mainly the code for splitting the flows into windows, lining up the label times with the pcap, and labeling the flows. I went through the code and understand what it does.